In [5]:
import pandas as pd

df_feat = pd.read_parquet("../data/df_feat.parquet")


In [6]:
import numpy as np
import pandas as pd

# 1) define columns you NEVER want as model predictors
ID_COLS = ["DUPERSID", "DUID", "PID", "PANEL", "VARSTR", "VARPSU"]
WEIGHT_COLS = ["LONGWT", "LSAQWT"]

# targets 
TARGET_COLS = ["TOTEXPY2", "LOG_TOTEXPY2", "HIGHCOST_Y2", "ANY_ED_Y2", "ANY_IP_Y2"]

EXCLUDE = set(ID_COLS + WEIGHT_COLS + TARGET_COLS)

# 2) candidate feature columns = everything else
feature_candidates = [c for c in df_feat.columns if c not in EXCLUDE]

# 3) categorical = object/category
cat_cols = df_feat[feature_candidates].select_dtypes(include=["object", "category"]).columns.tolist()

# 4) numeric = number types (int/float/bool)
num_cols = df_feat[feature_candidates].select_dtypes(include=[np.number]).columns.tolist()

print("n feature candidates:", len(feature_candidates))
print("n cat:", len(cat_cols))
print("n num:", len(num_cols))

cat_cols[:20], num_cols[:20]


n feature candidates: 128
n cat: 7
n num: 121


(['AGE_GROUP',
  'RACE_ETH',
  'REGIONY1_CAT',
  'EDU_GROUP',
  'POVCATY1_CAT',
  'FAMSIZE_Y1_GRP',
  'INS_TYPE_Y1'],
 ['YEARIND',
  'ALL5RDS',
  'DIED',
  'INST',
  'MILITARY',
  'ENTRSRVY',
  'LEFTUS',
  'OTHER',
  'AGEY1X',
  'AGEY2X',
  'AGELSTY1',
  'AGELSTY2',
  'SEX',
  'RACETHX',
  'HISPANX',
  'EDUCYR',
  'REGIONY1',
  'REGIONY2',
  'FAMINCY1',
  'FAMINCY2'])

In [10]:
TARGET_COLS = ["TOTEXPY2", "LOG_TOTEXPY2", "HIGHCOST_Y2", "ANY_ED_Y2", "ANY_IP_Y2"]

In [11]:
cat_cols = [
    "RACE_ETH",
    "REGIONY1_CAT",
    "EDU_GROUP",
    "POVCATY1_CAT",
    "FAMSIZE_Y1_GRP",
    "INS_TYPE_Y1",
]

In [12]:
#Numeric (use engineered columns, not raw)

num_cols = [
    # demographics / SES
    "AGE",
    "SEX_BIN",
    "LOG_FAMINCY1",
    "FAMSIZE_Y1",

   

    # employment
    "WORKED_Y1",
    "ANY_UNEMP_COMP_Y1",
    "LOG_UNEMP_COMP_Y1",
    "EMP_INFO_R12",
    "EMP_ATTACHED_ANY_R12_FILL0",  # model-friendly version

    # health status baseline
    "RTHLTH1_FAIRPOOR",
    "MNHLTH1_FAIRPOOR",

    # chronic conditions baseline
    "HIBPDXY1_BIN",
    "CHDDXY1_BIN",
    "STRKDXY1_BIN",
    "CHOLDXY1_BIN",
    "ASTHDXY1_BIN",
    "DIABDXY1_M18_BIN",
    # "MULTIMORBIDITY_Y1",   # optional (can remove if you keep all *_BIN)

    # baseline utilisation/cost
    "LOG_TOTEXPY1",
    "ANY_ED_Y1",
    "ANY_IP_Y1",
]

## regression

In [9]:
from src.models import run_regression_baseline, run_classification_baseline

In [14]:
# Regression (LOG_TOTEXPY2)
reg_res = run_regression_baseline(
    df_feat,
    target_col="LOG_TOTEXPY2",
    num_cols=num_cols,
    cat_cols=cat_cols,
    alpha=0.01,
    l1_ratio=0.5,
    
)

print("REG VALID:", reg_res.valid_metrics)



REG VALID: {'MAE_log': 1.6572547698657452, 'RMSE_log': 2.2768887072679838, 'R2': 0.5140598186235315}


1) Regression baseline (predict LOG_TOTEXPY2)

Validation metrics:

MAE_log ≈ 1.66

RMSE_log ≈ 2.28

R² ≈ 0.514

Interpretation

R² ~ 0.51 means the model explains about 51% of the variation in year-2 log expenditures on the validation set. That’s a decent baseline for MEPS costs (these are noisy/heavy-tailed).

MAE/RMSE are on the log scale, so they’re not “dollars”. 
Roughly:
an average absolute log error of 1.66 corresponds to a multiplicative factor of about exp(1.66) ≈ 5.3 in (1+cost) terms. That sounds large, but cost prediction is inherently hard.

✅ Conclusion: baseline regression is reasonable, and you have a good reference point.

In [15]:
# Classification
hc_res = run_classification_baseline(df_feat, target_col="HIGHCOST_Y2", num_cols=num_cols, cat_cols=cat_cols)
ed_res = run_classification_baseline(df_feat, target_col="ANY_ED_Y2", num_cols=num_cols, cat_cols=cat_cols)
ip_res = run_classification_baseline(df_feat, target_col="ANY_IP_Y2", num_cols=num_cols, cat_cols=cat_cols)

print("HC VALID:", hc_res.valid_metrics)


print("ED VALID:", ed_res.valid_metrics)


print("IP VALID:", ip_res.valid_metrics)


HC VALID: {'AUC': 0.8166049052740303, 'PR_AUC': 0.343884020637643, 'best_t': np.float64(0.6), 'best_F1': 0.41282565130260523}
ED VALID: {'AUC': 0.7385583294290878, 'PR_AUC': 0.3670506610771668, 'best_t': np.float64(0.65), 'best_F1': 0.4149377593360996}
IP VALID: {'AUC': 0.7902182993716174, 'PR_AUC': 0.2803399591324149, 'best_t': np.float64(0.7999999999999999), 'best_F1': 0.36607142857142855}


2) Classification baselines (3 tasks)
A) High-cost (HIGHCOST_Y2, top 10%)

AUC ≈ 0.817 (good discrimination)

PR_AUC ≈ 0.344 (reasonable for a 10% positive class)

Best threshold on validation: t ≈ 0.60

Best F1 ≈ 0.413

✅ Meaning: your model can rank high-cost people fairly well.

B) ED visit (ANY_ED_Y2, ~14% positive)

AUC ≈ 0.739 (moderate)

PR_AUC ≈ 0.367 (not bad given prevalence)

Best threshold: t ≈ 0.65

Best F1 ≈ 0.415

✅ Meaning: you can predict “any ED visit” somewhat, but it’s harder than high-cost.

C) Inpatient stay (ANY_IP_Y2, ~7% positive)

AUC ≈ 0.790 (good)

PR_AUC ≈ 0.280 (expected to be low because positives are rare)

Best threshold: t ≈ 0.80

Best F1 ≈ 0.366

✅ Meaning: IP is imbalanced, so PR_AUC and F1 are naturally lower; AUC still looks strong.

## Hyperparameter tuning (use validation or CV)

A) Regression: tune ElasticNet (alpha, l1_ratio)

Use GridSearchCV on the training set only, then evaluate on val.

In [16]:
from src.models import make_preprocess, split_train_val_test

In [23]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.metrics import  root_mean_squared_error,r2_score

# 1) build X, y
X = df_feat[num_cols + cat_cols]
y = df_feat["LOG_TOTEXPY2"]

# 2) split
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, random_state=42, stratify=False
)

# 3) build preprocess
preprocess = make_preprocess(num_cols, cat_cols, scale_numeric=True)

# 4) pipeline
reg_model = Pipeline([
    ("preprocess", preprocess),
    ("model", ElasticNet(max_iter=20000, random_state=42)),
])

# 5) grid search on TRAIN only
param_grid = {
    "model__alpha": np.logspace(-4, -1, 7),
    "model__l1_ratio": [0.1, 0.5, 0.9],
}

gs = GridSearchCV(
    reg_model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
)
gs.fit(X_train, y_train)

best_reg = gs.best_estimator_
print("Best params:", gs.best_params_)

# 6) evaluate on VAL
val_pred = best_reg.predict(X_val)
val_rmse = root_mean_squared_error(y_val, val_pred)
val_r2 = r2_score(y_val, val_pred)

print("VAL RMSE_log:", val_rmse)
print("VAL R2:", val_r2)


Best params: {'model__alpha': np.float64(0.001), 'model__l1_ratio': 0.9}
VAL RMSE_log: 2.2786428126197573
VAL R2: 0.513310797725858


你的 thesis 里可以把重点更自然地放在 risk stratification（分类） 上：高费用/急诊/住院风险识别；

回归可以作为“补充任务”，解释为什么更难（heavy tail, heterogeneity），并用更合适的模型提高一点即可。

## test result

In [24]:
# Regression
reg_res = run_regression_baseline(
    df_feat,
    target_col="LOG_TOTEXPY2",
    num_cols=num_cols,
    cat_cols=cat_cols,
    alpha=0.01,
    l1_ratio=0.5,
)

print("REG VALID:", reg_res.valid_metrics)
print("REG TEST:",  reg_res.test_metrics)   # ← 加这一行




REG VALID: {'MAE_log': 1.6572547698657452, 'RMSE_log': 2.2768887072679838, 'R2': 0.5140598186235315}
REG TEST: {'MAE_log': 1.5553245943844185, 'RMSE_log': 2.1653297844655826, 'R2': 0.5230276404428619}


In [25]:
# Classification (3 tasks)
hc_res = run_classification_baseline(df_feat, target_col="HIGHCOST_Y2", num_cols=num_cols, cat_cols=cat_cols)
ed_res = run_classification_baseline(df_feat, target_col="ANY_ED_Y2",   num_cols=num_cols, cat_cols=cat_cols)
ip_res = run_classification_baseline(df_feat, target_col="ANY_IP_Y2",   num_cols=num_cols, cat_cols=cat_cols)

print("HC VALID:", hc_res.valid_metrics)
print("HC TEST:",  hc_res.test_metrics)     # ← 加

print("ED VALID:", ed_res.valid_metrics)
print("ED TEST:",  ed_res.test_metrics)     # ← 加

print("IP VALID:", ip_res.valid_metrics)
print("IP TEST:",  ip_res.test_metrics)     # ← 加

HC VALID: {'AUC': 0.8166049052740303, 'PR_AUC': 0.343884020637643, 'best_t': np.float64(0.6), 'best_F1': 0.41282565130260523}
HC TEST: {'AUC': 0.8496756146009878, 'PR_AUC': 0.4238439255285054, 'F1_at_best_t': 0.44952380952380955}
ED VALID: {'AUC': 0.7385583294290878, 'PR_AUC': 0.3670506610771668, 'best_t': np.float64(0.65), 'best_F1': 0.4149377593360996}
ED TEST: {'AUC': 0.7066193695201125, 'PR_AUC': 0.33085901207493673, 'F1_at_best_t': 0.37333333333333335}
IP VALID: {'AUC': 0.7902182993716174, 'PR_AUC': 0.2803399591324149, 'best_t': np.float64(0.7999999999999999), 'best_F1': 0.36607142857142855}
IP TEST: {'AUC': 0.7506499847421422, 'PR_AUC': 0.21956377090688456, 'F1_at_best_t': 0.27722772277227725}


In [ ]:
import pandas as pd

rows = [
    {"Task": "Regression (LOG_TOTEXPY2)", "Split": "VALID", **reg_res.valid_metrics},
    {"Task": "Regression (LOG_TOTEXPY2)", "Split": "TEST",  **reg_res.test_metrics},

    {"Task": "High-cost (HIGHCOST_Y2)", "Split": "VALID", **hc_res.valid_metrics},
    {"Task": "High-cost (HIGHCOST_Y2)", "Split": "TEST",  **hc_res.test_metrics},
 
    {"Task": "Any ED (ANY_ED_Y2)", "Split": "VALID", **ed_res.valid_metrics},
    {"Task": "Any ED (ANY_ED_Y2)", "Split": "TEST",  **ed_res.test_metrics},

    {"Task": "Any IP (ANY_IP_Y2)", "Split": "VALID", **ip_res.valid_metrics},
    {"Task": "Any IP (ANY_IP_Y2)", "Split": "TEST",  **ip_res.test_metrics},
]

results_table = pd.DataFrame(rows)
results_table


,Task,Split,MAE_log,RMSE_log,R2,AUC,PR_AUC,best_t,best_F1,F1_at_best_t
0,Regression (LOG_TOTEXPY2),VALID,1.657255,2.276889,0.514060,NaN,NaN,NaN,NaN,NaN
1,Regression (LOG_TOTEXPY2),TEST,1.555325,2.165330,0.523028,NaN,NaN,NaN,NaN,NaN
2,High-cost (HIGHCOST_Y2),VALID,NaN,NaN,NaN,0.816605,0.343884,0.60,0.412826,NaN
3,High-cost (HIGHCOST_Y2),TEST,NaN,NaN,NaN,0.849676,0.423844,NaN,NaN,0.449524
4,Any ED (ANY_ED_Y2),VALID,NaN,NaN,NaN,0.738558,0.367051,0.65,0.414938,NaN
5,Any ED (ANY_ED_Y2),TEST,NaN,NaN,NaN,0.706619,0.330859,NaN,NaN,0.373333
6,Any IP (ANY_IP_Y2),VALID,NaN,NaN,NaN,0.790218,0.280340,0.80,0.366071,NaN
7,Any IP (ANY_IP_Y2),TEST,NaN,NaN,NaN,0.750650,0.219564,NaN,NaN,0.277228


住院预测最难，因为住院的人太少；模型能把高风险的人大致排在前面（AUC还可以），但要把概率变成0/1时，很难同时做到误报少又漏报少，所以PR-AUC和F1会比较低。